# 600 · Neural Network Building Blocks

This notebook covers `sc_flow.backends.torch.nn`.

| Class | Role |
|---|---|
| `MLP` | Multi-layer perceptron |
| `Resnet1d` | Residual network |
| `MLPUnconditionalVF` | Full velocity field |

In [7]:
import torch

from sc_flow.backends.torch.nn import MLP, Resnet1d, MLPUnconditionalVF

torch.manual_seed(0)

## 1. `MLP`

A fully-connected feedforward network. Used as the building block inside `MLPUnconditionalVF`.

```python
MLP(
    input_dim,                              # (int) input dimensionality
    output_dim,                             # (int) output dimensionality
    hidden_dims = None,                     # Sequence[int] — e.g. (128, 128); default: no hidden layers
    activation_cls = None,                  # hidden layer activation; default: torch.nn.ReLU
    final_activation_cls = None,            # output layer activation; default: torch.nn.Identity
    activation_cls_kwargs = None,           # kwargs forwarded to activation_cls
    final_activation_cls_kwargs = None,     # kwargs forwarded to final_activation_cls
    use_batchnorm = False,                  # BatchNorm1d after each linear layer
    batchnorm_eps = 1e-3,
    batchnorm_momentum = 1e-2,
    batchnorm_affine = True,
    batchnorm_track_running_stats = True,
    use_layernorm = False,                  # LayerNorm after each linear layer
    layernorm_eps = 1e-5,
    layernorm_elementwise_affine = False,
    layernorm_bias = False,
    dropout_p = 0.0,                        # dropout probability; 0.0 = disabled
    dropout_inplace = False,
    bias = True,                            # learn bias in linear layers
)
```

In [8]:
# Basic MLP: 8 → (64, 64) → 4
mlp = MLP(
    input_dim=8,
    output_dim=4,
    hidden_dims=(64, 64),
    activation_cls=torch.nn.GELU,
    use_layernorm=True,
    dropout_p=0.1,
)
print(mlp._mlp)

x = torch.randn(32, 8)  # batch of 32 samples, 8-dim input
y = mlp(x)
print(f"\nInput shape:  {x.shape}  →  Output shape: {y.shape}")

Sequential(
  (layer_0): Sequential(
    (linear): Linear(in_features=8, out_features=64, bias=True)
    (layernorm): LayerNorm((64,), eps=1e-05, elementwise_affine=False)
    (activation): GELU(approximate='none')
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (layer_1): Sequential(
    (linear): Linear(in_features=64, out_features=64, bias=True)
    (layernorm): LayerNorm((64,), eps=1e-05, elementwise_affine=False)
    (activation): GELU(approximate='none')
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (layer_2): Sequential(
    (linear): Linear(in_features=64, out_features=4, bias=True)
    (layernorm): LayerNorm((4,), eps=1e-05, elementwise_affine=False)
    (activation): Identity()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

Input shape:  torch.Size([32, 8])  →  Output shape: torch.Size([32, 4])


In [9]:
print(f"Params: {sum(p.numel() for p in mlp.parameters()):,}")

Params: 4,996


## 2. `Resnet1d`

A conditional residual network. Used inside `MLPUnconditionalVF` when `conditioning_id='resnet1d'` — you won't instantiate it directly.

```python
Resnet1d(
    input_dim,                              # (int) state dimensionality
    embedding_dim,                          # (int) conditioning vector dimensionality
    num_resnet_layers = None,               # number of residual blocks; default: 3
    output_dim = None,                      # output dim; default: input_dim (state dim unchanged)
    activation_cls = None,                  # default: torch.nn.SiLU
    activation_cls_kwargs = None,
    use_batchnorm = False,
    batchnorm_eps = 1e-3,
    batchnorm_momentum = 1e-2,
    batchnorm_affine = True,
    batchnorm_track_running_stats = True,
    use_layernorm = False,
    layernorm_eps = 1e-5,
    layernorm_elementwise_affine = False,
    layernorm_bias = False,
    dropout_p = 0.0,
    dropout_inplace = False,
    bias = True,
)
```

In [10]:
# Resnet1d: state_dim=16, conditioned on a 32-dim embedding
resnet = Resnet1d(
    input_dim=16,
    embedding_dim=32,
    num_resnet_layers=3,
    output_dim=16,  # keeps state dim
    activation_cls=torch.nn.SiLU,
)
print(resnet._resnet)

x_state = torch.randn(32, 16)  # (batch, state_dim)
cond = torch.randn(32, 32)  # (batch, embedding_dim)

out = resnet(x_state, cond)  # Resnet1d.forward takes (x, cond)
print(f"\nState {x_state.shape}  + Cond {cond.shape}  →  {out.shape}")

Sequential(
  (layer_0): ModuleDict(
    (net1): Sequential(
      (activation): SiLU()
      (linear): Linear(in_features=16, out_features=16, bias=True)
    )
    (cond_proj): Sequential(
      (0): SiLU()
      (1): Linear(in_features=32, out_features=16, bias=True)
    )
    (net2): Sequential(
      (activation): SiLU()
      (linear): Linear(in_features=16, out_features=16, bias=True)
    )
    (skip_proj): Identity()
  )
  (layer_1): ModuleDict(
    (net1): Sequential(
      (activation): SiLU()
      (linear): Linear(in_features=16, out_features=16, bias=True)
    )
    (cond_proj): Sequential(
      (0): SiLU()
      (1): Linear(in_features=32, out_features=16, bias=True)
    )
    (net2): Sequential(
      (activation): SiLU()
      (linear): Linear(in_features=16, out_features=16, bias=True)
    )
    (skip_proj): Identity()
  )
  (layer_2): ModuleDict(
    (net1): Sequential(
      (activation): SiLU()
      (linear): Linear(in_features=16, out_features=16, bias=True)
    )

## 3. `MLPUnconditionalVF`

The full velocity field. Assembles time featurisation → time encoder → state encoder → conditioning layer → VF decoder. We instantiate this

Most parameters default to `None`. When `None`, the init has default values for these parameters

```python
MLPUnconditionalVF(
    state_dim,                              # (int) dimensionality of the state space

    # ── Time featurisation ───────────────────────────────────────────────────
    time_features_id = None,               # None = raw scalar t | "torch-cfm" = sinusoidal embeddings
    time_features_fn = None,               # custom featuriser fn — takes precedence over time_features_id
    num_time_features = None,              # None → 256 ## WHY IS THE GLOBAL CONSTANT NOT CALLED DIRECTLY INSTEAD OF PUTTING IT IN A FUNC(). ONLY TO DEFAULT TO 1 OR 256 THEN?
    max_period = None,                     # None → 1000  (sinusoidal only)
    time_features_kwargs = None,           # extra kwargs for custom time_features_fn

    # ── Encoders ─────────────────────────────────────────────────────────────
    encode_state = True,
    encode_time = True,
    state_encoder_output_dim = None,       # None → 32
    time_encoder_output_dim = None,        # None → 16
    state_encoder_mlp_kwargs = None,       # MLP kwargs for state encoder; None → {}
    time_encoder_mlp_kwargs = None,        # MLP kwargs for time encoder; None → {}

    # ── Conditioning layer ───────────────────────────────────────────────────
    conditioning_id = None,                # None → "concat" | "resnet1d"
    conditioning_fn = None,               # custom conditioning callable — takes precedence over conditioning_id
    conditioning_kwargs = None,            # extra kwargs; for "resnet1d": num_resnet_layers, etc.; None → {}

    # ── VF decoder ───────────────────────────────────────────────────────────
    vf_decoder_mlp_kwargs = None,          # MLP kwargs for the output decoder; None → {}

    # ── Perturbation condition encoder (conditional VF only) ─────────────────
    condition_encoder_input_layers = None, # None = unconditional
                                           # dict[str, dict] → conditional, e.g.:
                                           # {"cytokine": {"input_dim": 2560, "hidden_dims": (256,), "output_dim": 128}}
    condition_encoder_output_dim = None,   # None → 32
    condition_encoder_pooling_mode = "mean",  # "mean" | "sum"
    condition_encoder_pooling_kwargs = None,
    condition_encoder_output_layers_kwargs = None,

    # ── Source encoder (cell-to-cell transport only) ──────────────────────────
    source_encoder_mlp_kwargs = None,      # None = disabled; pass MLP kwargs to enable
    source_encoder_output_dim = None,      # None → 16
)
```

**Forward / inference:**

```python
# # Training
# v = vf(t, xt)                                        # unconditional
# v = vf(t, xt, condition_dict={"cytokine": emb})      # conditional
# v = vf(t, xt, source=src_cells)                      # with source encoder

# # ODE solving — bakes condition into a plain (t, x) → v callable
# vf_fn = vf.get_vf_fn(condition_dict={"cytokine": emb}, source=src_cells)
# v = vf_fn(t, xt)
```

In [ ]:
# Unconditional VF for 2-D toy data
vf_uncond = MLPUnconditionalVF(
    state_dim=2,
    encode_state=True,
    encode_time=True,
    time_features_id="torch-cfm",  # sinusoidal time embeddings
    state_encoder_mlp_kwargs={"hidden_dims": (32,)},
    time_encoder_mlp_kwargs={"hidden_dims": (16,)},
    vf_decoder_mlp_kwargs={"hidden_dims": (64, 64)},
)
print(f"Params: {sum(p.numel() for p in vf_uncond.parameters()):,}")

# Forward pass
batch_size = 64
t = torch.rand(batch_size)  # scalar time per sample
xt = torch.randn(batch_size, 2)  # current state

v = vf_uncond(t, xt)  # → (batch, 2) velocity
print(f"output: {v.shape}")

# # get_vf_fn() returns (t, x) → v for use with ODESolver
# vf_fn = vf_uncond.get_vf_fn()
t_scalar = torch.tensor(0.5)  # ODESolver passes a scalar t
# v_fn = vf_fn(t_scalar, xt)

Params: 12,962
output: torch.Size([64, 2])


In [12]:
# Unconditional VF with Resnet1d conditioning layer
# (state + time are conditioned via residual blocks instead of concatenation)
vf_resnet = MLPUnconditionalVF(
    state_dim=2,
    time_features_id="torch-cfm",
    conditioning_id="resnet1d",  # use Resnet1d instead of concat
    conditioning_kwargs={"num_resnet_layers": 3},
    vf_decoder_mlp_kwargs={"hidden_dims": (64, 64)},
)
v_res = vf_resnet(t, xt)
print(f"Resnet-conditioned VF output: {v_res.shape}")

Resnet-conditioned VF output: torch.Size([64, 2])


## 4. `MLPUnconditionalVF` — Conditional

To add a **perturbation covariate** (e.g. a cytokine ESM2 embedding), pass
`condition_encoder_input_layers` — a `dict[str, dict]` mapping each covariate name
to MLP kwargs for its encoder:

```python
condition_encoder_input_layers = {
    "cytokine": {"input_dim": 2560, "hidden_dims": (256,), "output_dim": 128},
}
```

At inference the condition is passed as `condition_dict={"cytokine": tensor}`.

In [13]:
COND_DIM = 16  # e.g. a 16-dim perturbation embedding

vf_cond = MLPUnconditionalVF(
    state_dim=2,
    time_features_id="torch-cfm",
    vf_decoder_mlp_kwargs={"hidden_dims": (64, 64)},
    # ── condition encoder ──────────────────────────────────────
    condition_encoder_input_layers={
        "perturbation": {
            "input_dim": COND_DIM,
            "hidden_dims": (32,),
            "output_dim": 16,
        }
    },
    condition_encoder_output_dim=16,
)
print(f"Is conditional: {vf_cond.is_conditional}")
print(f"Params:         {sum(p.numel() for p in vf_cond.parameters()):,}")

# Forward pass with condition
cond_batch = torch.randn(batch_size, COND_DIM)
condition_dict = {"perturbation": cond_batch}

v_cond = vf_cond(t, xt, condition_dict=condition_dict)
print(f"Conditional velocity: {v_cond.shape}")

Is conditional: True
Params:         14,002
Conditional velocity: torch.Size([64, 2])


In [27]:
# get_vf_fn() for conditional VF: bakes the condition into the callable
# so ODESolver only needs to call (t, x)
vf_fn_cond = vf_cond.get_vf_fn(condition_dict=condition_dict)

v_cond_fn = vf_fn_cond(t_scalar, xt)
print(f"Conditional vf_fn output: {v_cond_fn.shape}")

Conditional vf_fn output: torch.Size([64, 2])


In [15]:
# Source encoder: retain info about the source cells (for cell-to-cell transport)
vf_with_src = MLPUnconditionalVF(
    state_dim=2,
    time_features_id="torch-cfm",
    vf_decoder_mlp_kwargs={"hidden_dims": (64, 64)},
    source_encoder_mlp_kwargs={"hidden_dims": (32,)},  # enables source encoding
    source_encoder_output_dim=16,
)
print(f"Uses source encoder: {vf_with_src.use_source_encoder}")

source_cells = torch.randn(batch_size, 2)
v_src = vf_with_src(t, xt, source=source_cells)
print(f"Source-conditioned velocity: {v_src.shape}")

Uses source encoder: True
Source-conditioned velocity: torch.Size([64, 2])
